# XGBoost for Cross-Sectional Return Prediction

## Objective

Test whether allowing nonlinear relationships and feature interactions
improves next-day cross-sectional equity return prediction relative to
regularized linear models.

The linear-model benchmark from Notebook 03 showed:

- essentially zero out-of-sample point-forecast R²,
- weak but positive validation Rank IC,
- improved ranking performance under strong regularization.

XGBoost is introduced as a nonlinear challenger.

The central research question is:

> Do nonlinearities and interactions provide incremental out-of-sample predictive information beyond regularized linear models?

Model fitting uses only the training period (2010–2018).

Hyperparameters are selected using the validation period (2019–2021).

The test period (2022 onward) remains untouched.

In [3]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

from sklearn.metrics import mean_squared_error

from xgboost import XGBRegressor

In [4]:
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

df = pd.read_csv(
    DATA_DIR / "alpha_dataset.csv",
    parse_dates=["date"]
)

df.head()

,date,adj_close,close,high,low,open,volume,ticker,ret_1d,next_obs_date,...,beta_60,ret_1d_z,mom_5_z,mom_20_z,mom_60_z,vol_20_z,vol_60_z,price_ma20_gap_z,dollar_volume_z20_z,beta_60_z
0,2010-03-31,7.028762,8.392857,8.450357,8.373571,8.410357,430659600,AAPL,-0.003604,2010-04-01,...,1.497634,0.072740,1.577808,1.472380,0.625249,0.258525,0.577985,1.687521,-0.628822,1.199474
1,2010-03-31,39.938347,59.840000,60.220001,59.689999,60.000000,4975100,AMGN,-0.004160,2010-04-01,...,0.439358,-0.004035,0.822124,-0.028272,-0.077191,-0.257294,-0.873835,0.610111,-0.222617,-1.426405
2,2010-03-31,56.966702,72.610001,73.059998,71.949997,72.919998,5884600,BA,-0.012512,2010-04-01,...,1.289498,-1.156854,0.229580,1.556568,2.951779,0.651530,0.829966,1.036899,-0.170435,0.683031
3,2010-03-31,13.703833,17.850000,17.980000,17.670000,17.690001,114280900,BAC,0.005068,2010-04-01,...,1.586340,1.269817,1.012583,0.791000,1.086440,0.173627,1.174258,1.504379,-0.855187,1.419579
4,2010-03-31,29.604021,40.500000,41.099998,40.099998,40.799999,55035890,C,-0.009780,2010-04-01,...,1.610078,-0.779787,-1.615726,2.922016,1.691174,3.920720,2.263978,0.335032,0.075735,1.478478


In [5]:
MODEL_FEATURES = [
    "ret_1d_z",
    "mom_5_z",
    "mom_20_z",
    "mom_60_z",
    "vol_20_z",
    "vol_60_z",
    "price_ma20_gap_z",
    "dollar_volume_z20_z",
    "beta_60_z",
]

TARGET = "target_cs_1d"

In [6]:
missing_cols = (
    set(MODEL_FEATURES + [TARGET])
    - set(df.columns)
)

assert not missing_cols, (
    f"Missing columns: {missing_cols}"
)

In [7]:
train = (
    df[df["split"] == "train"]
    .sort_values(["date", "ticker"])
    .reset_index(drop=True)
)

validation = (
    df[df["split"] == "validation"]
    .sort_values(["date", "ticker"])
    .reset_index(drop=True)
)

test = (
    df[df["split"] == "test"]
    .sort_values(["date", "ticker"])
    .reset_index(drop=True)
)

In [8]:
X_train = train[MODEL_FEATURES]
y_train = train[TARGET]

X_val = validation[MODEL_FEATURES]
y_val = validation[TARGET]

In [9]:
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test rows:", len(test))

print()
print(
    "Missing train values:",
    X_train.isna().sum().sum()
)

print(
    "Missing validation values:",
    X_val.isna().sum().sum()
)

Train: (66120, 9)
Validation: (22710, 9)
Test rows: 34950

Missing train values: 0
Missing validation values: 0


## Evaluation functions (same as notebook 03)

In [10]:
def oos_r2(y_true, y_pred):
    """
    Out-of-sample R² relative to predicting
    zero cross-sectional return.
    """

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    model_sse = np.sum(
        (y_true - y_pred) ** 2
    )

    zero_sse = np.sum(
        y_true ** 2
    )

    return 1 - model_sse / zero_sse

In [11]:
def daily_ic(
    df,
    prediction_col,
    target_col=TARGET,
    method="spearman",
):
    """
    Calculate cross-sectional prediction/return
    correlation separately for every date.
    """

    dates = []
    ic_values = []

    for date, d in df.groupby("date"):

        valid = d[
            [prediction_col, target_col]
        ].dropna()

        if len(valid) < 5:
            continue

        if (
            valid[prediction_col].nunique() < 2
            or valid[target_col].nunique() < 2
        ):
            ic = np.nan

        else:
            ic = valid[prediction_col].corr(
                valid[target_col],
                method=method,
            )

        dates.append(date)
        ic_values.append(ic)

    return pd.Series(
        ic_values,
        index=pd.to_datetime(dates),
        name=f"{method}_ic",
    )

In [12]:
def evaluate_predictions(
    data,
    predictions,
    model_name,
):
    """
    Evaluate predictions using:
    - MSE
    - OOS R²
    - Pearson IC
    - Rank IC
    """

    temp = data[
        ["date", "ticker", TARGET]
    ].copy()

    temp["prediction"] = predictions

    mse = mean_squared_error(
        temp[TARGET],
        temp["prediction"],
    )

    r2 = oos_r2(
        temp[TARGET],
        temp["prediction"],
    )

    pearson_ic = daily_ic(
        temp,
        prediction_col="prediction",
        method="pearson",
    )

    rank_ic = daily_ic(
        temp,
        prediction_col="prediction",
        method="spearman",
    )

    return {
        "model": model_name,
        "mse": mse,
        "oos_r2": r2,
        "mean_pearson_ic":
            pearson_ic.mean(),
        "mean_rank_ic":
            rank_ic.mean(),
        "rank_ic_std":
            rank_ic.std(),
        "pct_rank_ic_positive":
            (rank_ic > 0).mean(),
    }

---

# XGBoost

XGBoost builds many small decision trees sequentially.

A single decision tree might learn a rule such as:

    mom_5_z < -1.2
        |
        +-- yes -> positive forecast
        |
        +-- no
              |
              vol_20_z > 0.8
                  |
                  +-- yes -> negative forecast
                  +-- no  -> approximately zero

A single tree is usually weak.

Boosting constructs a sequence:

    prediction =
        tree_1
        + learning_rate * tree_2
        + learning_rate * tree_3
        + ...

Each new tree attempts to correct errors made by the existing ensemble.

Unlike linear regression, trees can naturally represent:

- nonlinear thresholds,
- interactions between variables,
- different behavior in different regions of feature space.

The risk is overfitting, especially because next-day equity returns
contain very little signal. We therefore use shallow trees and a small
hyperparameter search.

## Define a conservative XGBoost baseline

Our problem has: $$ \text{signal} \ll \text{noise} $$
So we want to make it relatively difficult for XGBoost to memorize the training sample.

In particular, max_depth = 2 means that each tree is very simple.

In [13]:
xgb_baseline = XGBRegressor(
    objective="reg:squarederror",

    # Keep trees shallow because signal is weak
    max_depth=2,

    # Small learning rate
    learning_rate=0.05,

    # Number of boosting trees
    n_estimators=200,

    # Randomly use 80% of observations per tree
    subsample=0.8,

    # Randomly use 80% of features per tree
    colsample_bytree=0.8,

    # Minimum amount of data/effective weight
    # needed in child nodes
    min_child_weight=10,

    # L2 regularization
    reg_lambda=10.0,

    # No L1 penalty initially
    reg_alpha=0.0,

    random_state=42,

    n_jobs=-1,
)

In [14]:
xgb_baseline.fit(
    X_train,
    y_train
)

,"objective objective: typing.Union[str, xgboost.objective.Objective, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets im

## Baseline XGBoost predictions

In [15]:
xgb_train_pred = (
    xgb_baseline.predict(
        X_train
    )
)

xgb_val_pred = (
    xgb_baseline.predict(
        X_val
    )
)

In [16]:
baseline_train_metrics = (
    evaluate_predictions(
        train,
        xgb_train_pred,
        "XGBoost Baseline - Train",
    )
)

baseline_val_metrics = (
    evaluate_predictions(
        validation,
        xgb_val_pred,
        "XGBoost Baseline - Validation",
    )
)

pd.DataFrame([
    baseline_train_metrics,
    baseline_val_metrics,
])

,model,mse,oos_r2,mean_pearson_ic,mean_rank_ic,rank_ic_std,pct_rank_ic_positive
0,XGBoost Baseline - Train,0.000119,0.006863,0.076433,0.057612,0.222649,0.598911
1,XGBoost Baseline - Validation,0.000226,-0.000553,0.015435,0.009514,0.248099,0.501982


## Define a small hyperparameter grid
We are only going to tune three important complexity controls to avoid overfitting the validation set

We get
$$ \text{Max Depth} \times \text{Learning Rates} \times \text{N Estimators} = 3 \times 2 \times 2 = 12 $$
models

In [ ]:
MAX_DEPTHS = [ 1, 2, 3 ]

LEARNING_RATES = [ 0.02, 0.05 ]

N_ESTIMATORS = [ 100, 300 ]

In [18]:
# Hyperparameter search
xgb_results = []
xgb_models = {}

for max_depth in MAX_DEPTHS:

    for learning_rate in LEARNING_RATES:

        for n_estimators in N_ESTIMATORS:

            model = XGBRegressor(
                objective="reg:squarederror",

                max_depth=max_depth,
                learning_rate=learning_rate,
                n_estimators=n_estimators,

                subsample=0.8,
                colsample_bytree=0.8,

                min_child_weight=10,

                reg_lambda=10.0,
                reg_alpha=0.0,

                random_state=42,
                n_jobs=-1,
            )

            # Train using only 2010-2018
            model.fit(
                X_train,
                y_train
            )

            # Evaluate on 2019-2021
            val_pred = model.predict(
                X_val
            )

            metrics = evaluate_predictions(
                validation,
                val_pred,
                model_name=(
                    f"XGB depth={max_depth}, "
                    f"lr={learning_rate}, "
                    f"trees={n_estimators}"
                ),
            )

            metrics["max_depth"] = max_depth
            metrics["learning_rate"] = learning_rate
            metrics["n_estimators"] = n_estimators

            xgb_results.append(
                metrics
            )

            xgb_models[
                (
                    max_depth,
                    learning_rate,
                    n_estimators,
                )
            ] = model

In [19]:
# Convert to dataframe
xgb_results = pd.DataFrame(
    xgb_results
)

xgb_results = (
    xgb_results
    .sort_values(
        "mean_rank_ic",
        ascending=False
    )
    .reset_index(drop=True)
)

xgb_results

,model,mse,oos_r2,mean_pearson_ic,mean_rank_ic,rank_ic_std,pct_rank_ic_positive,max_depth,learning_rate,n_estimators
0,"XGB depth=1, lr=0.05, trees=300",0.000226,0.000356,0.025115,0.016245,0.263298,0.516513,1,0.05,300
1,"XGB depth=1, lr=0.02, trees=300",0.000226,0.000290,0.021564,0.015580,0.270179,0.527081,1,0.02,300
2,"XGB depth=3, lr=0.02, trees=100",0.000226,0.000124,0.021495,0.014081,0.261821,0.528402,3,0.02,100
3,"XGB depth=1, lr=0.02, trees=100",0.000226,0.000205,0.018998,0.012782,0.267576,0.508587,1,0.02,100
4,"XGB depth=2, lr=0.02, trees=300",0.000226,-0.000097,0.017212,0.010772,0.262753,0.524439,2,0.02,300
5,"XGB depth=2, lr=0.02, trees=100",0.000226,0.000162,0.018824,0.009340,0.267667,0.528402,2,0.02,100
6,"XGB depth=2, lr=0.05, trees=100",0.000226,-0.000226,0.015164,0.007780,0.255677,0.531044,2,0.05,100
7,"XGB depth=2, lr=0.05, trees=300",0.000226,-0.001019,0.012522,0.007225,0.237233,0.512550,2,0.05,300
8,"XGB depth=3, lr=0.02, trees=300",0.000226,-0.000654,0.015555,0.006973,0.247572,0.503303,3,0.02,300
9,"XGB depth=1, lr=0.05, trees=100",0.000226,0.000220,0.019206,0.006017,0.262758,0.515192,1,0.05,100


In [20]:
# Select XGBoost using validation Rank IC
best_depth = int(
    xgb_results.loc[
        0,
        "max_depth"
    ]
)

best_lr = float(
    xgb_results.loc[
        0,
        "learning_rate"
    ]
)

best_n_estimators = int(
    xgb_results.loc[
        0,
        "n_estimators"
    ]
)

In [21]:
# Print
print(
    "Best max_depth:",
    best_depth
)

print(
    "Best learning_rate:",
    best_lr
)

print(
    "Best n_estimators:",
    best_n_estimators
)

Best max_depth: 1
Best learning_rate: 0.05
Best n_estimators: 300


In [22]:
# Retrieve
best_xgb = xgb_models[
    (
        best_depth,
        best_lr,
        best_n_estimators,
    )
]

## Evaluate selected XGBoost on train and validation

In [23]:
best_xgb_train_pred = (
    best_xgb.predict(
        X_train
    )
)

best_xgb_val_pred = (
    best_xgb.predict(
        X_val
    )
)

In [24]:
# Metrics
best_xgb_train_metrics = (
    evaluate_predictions(
        train,
        best_xgb_train_pred,
        "XGBoost - Train",
    )
)

best_xgb_val_metrics = (
    evaluate_predictions(
        validation,
        best_xgb_val_pred,
        "XGBoost - Validation",
    )
)

pd.DataFrame([
    best_xgb_train_metrics,
    best_xgb_val_metrics,
])

,model,mse,oos_r2,mean_pearson_ic,mean_rank_ic,rank_ic_std,pct_rank_ic_positive
0,XGBoost - Train,0.000120,0.002304,0.044617,0.036045,0.232323,0.556715
1,XGBoost - Validation,0.000226,0.000356,0.025115,0.016245,0.263298,0.516513


## Load final linear predictions and add XGBoost predictions

In [25]:
linear_predictions = pd.read_csv(
    DATA_DIR
    / "linear_model_validation_predictions.csv",
    parse_dates=["date"],
)

linear_predictions.head()

,date,ticker,fwd_ret_1d,target_cs_1d,ols_pred,ridge_pred,elastic_net_pred
0,2019-01-02,AAPL,-0.099607,-0.075323,-0.000047,-0.000034,-0.000079
1,2019-01-02,AMGN,-0.015216,0.009068,-0.000126,-0.000010,-0.000010
2,2019-01-02,BA,-0.039869,-0.015585,-0.000186,-0.000077,-0.000141
3,2019-01-02,BAC,-0.016026,0.008258,-0.000493,-0.000120,-0.000168
4,2019-01-02,C,-0.018121,0.006163,-0.000471,-0.000124,-0.000175


In [26]:
xgb_predictions = validation[
    [
        "date",
        "ticker",
        "target_cs_1d",
        "fwd_ret_1d",
    ]
].copy()

xgb_predictions[
    "xgboost_pred"
] = best_xgb_val_pred

In [27]:
# Merge
all_validation_predictions = (
    linear_predictions.merge(
        xgb_predictions[
            [
                "date",
                "ticker",
                "xgboost_pred",
            ]
        ],
        on=[
            "date",
            "ticker",
        ],
        how="inner",
    )
)

In [28]:
# Check
all_validation_predictions.head()

,date,ticker,fwd_ret_1d,target_cs_1d,ols_pred,ridge_pred,elastic_net_pred,xgboost_pred
0,2019-01-02,AAPL,-0.099607,-0.075323,-0.000047,-0.000034,-0.000079,-0.000051
1,2019-01-02,AMGN,-0.015216,0.009068,-0.000126,-0.000010,-0.000010,0.000142
2,2019-01-02,BA,-0.039869,-0.015585,-0.000186,-0.000077,-0.000141,-0.000171
3,2019-01-02,BAC,-0.016026,0.008258,-0.000493,-0.000120,-0.000168,-0.000323
4,2019-01-02,C,-0.018121,0.006163,-0.000471,-0.000124,-0.000175,-0.000598


## Direct linear vs nonlinear comparison

In [32]:
comparison = []

# OLS
comparison.append(
    evaluate_predictions(
        validation,
        all_validation_predictions[
            "ols_pred"
        ].values,
        "OLS",
    )
)

# Ridge
comparison.append(
    evaluate_predictions(
        validation,
        all_validation_predictions[
            "ridge_pred"
        ].values,
        "Ridge",
    )
)

# Elastic Net
comparison.append(
    evaluate_predictions(
        validation,
        all_validation_predictions[
            "elastic_net_pred"
        ].values,
        "Elastic Net",
    )
)

# XGBoost
comparison.append(
    evaluate_predictions(
        validation,
        all_validation_predictions[
            "xgboost_pred"
        ].values,
        "XGBoost",
    )
)

In [33]:
# Table
comparison = pd.DataFrame(
    comparison
)

comparison.sort_values(
    "mean_rank_ic",
    ascending=False
)

,model,mse,oos_r2,mean_pearson_ic,mean_rank_ic,rank_ic_std,pct_rank_ic_positive
3,XGBoost,0.000226,0.000356,0.025115,0.016245,0.263298,0.516513
1,Ridge,0.000226,-0.000021,0.002429,0.009774,0.304946,0.531044
2,Elastic Net,0.000226,-0.000074,-0.001553,0.008646,0.309890,0.520476
0,OLS,0.000226,-0.000110,0.000670,0.003027,0.286216,0.523118


## Feature importance

We inspect XGBoost to see which features it actually uses most.

Use gain-based importance from the underlying booster

In [34]:
booster = best_xgb.get_booster()

gain_scores = booster.get_score(
    importance_type="gain"
)

In [35]:
# Construct Table
importance = pd.DataFrame({
    "feature": MODEL_FEATURES
})

# Map scores
importance["gain"] = (
    importance["feature"]
    .map(gain_scores)
    .fillna(0)
)

# Normalize
importance["gain_share"] = (
    importance["gain"]
    / importance["gain"].sum()
)

# Sort
importance = (
    importance
    .sort_values(
        "gain_share",
        ascending=False
    )
    .reset_index(drop=True)
)

importance

,feature,gain,gain_share
0,ret_1d_z,0.001019,0.132391
1,vol_20_z,0.000944,0.122665
2,vol_60_z,0.000897,0.116549
3,mom_20_z,0.000888,0.115369
4,mom_60_z,0.000814,0.105732
5,mom_5_z,0.000808,0.105001
6,beta_60_z,0.000790,0.102610
7,dollar_volume_z20_z,0.000778,0.101115
8,price_ma20_gap_z,0.000759,0.098566


In [36]:
# Plot the feature importance
fig = px.bar(
    importance.sort_values(
        "gain_share"
    ),
    x="gain_share",
    y="feature",
    orientation="h",
    title="XGBoost Feature Importance — Gain",
)

fig.show()

A large gain value means XGBoost found the feature useful for reducing prediction error.

## Rolling Rank IC

In [37]:
# We repeat the stability diagnostic similar to notebook 03
prediction_cols = {
    "OLS": "ols_pred",
    "Ridge": "ridge_pred",
    "Elastic Net": "elastic_net_pred",
    "XGBoost": "xgboost_pred",
}

In [38]:
# Calculate daily IC
ic_frames = []

for model_name, pred_col in prediction_cols.items():

    ic = daily_ic(
        all_validation_predictions,
        prediction_col=pred_col,
        method="spearman",
    )

    temp = pd.DataFrame({
        "date": ic.index,
        "rank_ic": ic.values,
        "model": model_name,
    })

    ic_frames.append(temp)


rank_ic_series = pd.concat(
    ic_frames,
    ignore_index=True
)

In [39]:
# Rolling 60-day mean
rank_ic_series = (
    rank_ic_series
    .sort_values(
        ["model", "date"]
    )
)

rank_ic_series[
    "rank_ic_60d"
] = (
    rank_ic_series
    .groupby("model")["rank_ic"]
    .transform(
        lambda s:
        s.rolling(
            60,
            min_periods=20
        ).mean()
    )
)

In [40]:
# Plot
fig = px.line(
    rank_ic_series,
    x="date",
    y="rank_ic_60d",
    color="model",
    title=(
        "Validation-Period "
        "60-Day Rolling Rank IC"
    ),
)

fig.add_hline(
    y=0,
    line_dash="dash"
)

fig.show()

## Prediction correlation

In [41]:
pred_corr = (
    all_validation_predictions[
        [
            "ols_pred",
            "ridge_pred",
            "elastic_net_pred",
            "xgboost_pred",
        ]
    ]
    .corr()
)

pred_corr

,ols_pred,ridge_pred,elastic_net_pred,xgboost_pred
ols_pred,1.000000,0.889149,0.828577,0.354672
ridge_pred,0.889149,1.000000,0.939548,0.372465
elastic_net_pred,0.828577,0.939548,1.000000,0.386451
xgboost_pred,0.354672,0.372465,0.386451,1.000000


In [42]:
fig = px.imshow(
    pred_corr,
    text_auto=".2f",
    aspect="auto",
    title=(
        "Correlation Between Model Predictions"
    ),
)

fig.show()

## Save XGBoost hyperparameters and combined validation predictions

In [43]:
xgb_params = {
    "max_depth":
        int(best_depth),

    "learning_rate":
        float(best_lr),

    "n_estimators":
        int(best_n_estimators),

    "subsample":
        0.8,

    "colsample_bytree":
        0.8,

    "min_child_weight":
        10,

    "reg_lambda":
        10.0,

    "reg_alpha":
        0.0,
}

In [ ]:
# Save
with open(
    DATA_DIR / "xgboost_params.json",
    "w"
) as f:

    json.dump(
        xgb_params,
        f,
        indent=4,
    )

In [45]:
# Inspect
xgb_params

{'max_depth': 1,
 'learning_rate': 0.05,
 'n_estimators': 300,
 'subsample': 0.8,
 'colsample_bytree': 0.8,
 'min_child_weight': 10,
 'reg_lambda': 10.0,
 'reg_alpha': 0.0}

In [46]:
all_validation_predictions.to_csv(
    DATA_DIR
    / "model_validation_predictions.csv",
    index=False,
)

## Findings

XGBoost produced the strongest validation-period predictive performance
among the models tested.

The selected specification used depth-1 trees, a learning rate of 0.05,
and 300 boosting rounds. Its validation Rank IC was approximately 0.016,
compared with approximately 0.010 for the best Ridge model.

Importantly, the selected tree depth was only one. Therefore the improvement
does not provide evidence that complex feature interactions were necessary.
Instead, the result suggests that allowing nonlinear threshold relationships
within individual predictors provided incremental predictive information
beyond a purely linear specification.

Increasing tree depth generally reduced validation performance, with the
most complex specifications substantially overfitting the noisy return target.

XGBoost was also the first model to achieve positive validation out-of-sample
R², although the magnitude remained extremely small. This reinforces that
short-horizon equity returns are overwhelmingly noisy and that the main
predictive value of the models lies in relative cross-sectional ranking
rather than accurate return-magnitude forecasts.

Feature gain importance was broadly distributed across the nine predictors,
with no single feature dominating the model.

The economic value of the predictive signal remains unknown. The next step
is to evaluate the frozen model specifications on the untouched test period
using walk-forward estimation and a long-short portfolio after turnover and
transaction costs.